In [1]:
# SYS Libraries ##########################################################################################################
import sys
import os
import subprocess
import glob

# Basic Libraries ##########################################################################################################
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import scipy
from skimage import io
from skimage import exposure
from scipy import ndimage
from IPython.display import display

# Image Manipulation Libraries ##########################################################################################################
import cv2
from shapely.geometry import MultiPolygon, Polygon, Point, LinearRing
import shapely, shapely.geometry
import shapely.wkt
import shapely.affinity

import csv

#import tifffile as tiff

#from PIL import Image, ImageFilter, ImageEnhance, ImageOps, ImageChops

# GDAL Libraries ##########################################################################################################
from osgeo import gdal
from osgeo import gdal_array
from osgeo import osr


from sklearn.metrics import jaccard_similarity_score
from shapely.geometry import MultiPolygon, Polygon
import shapely.wkt
import shapely.affinity
from collections import defaultdict

%matplotlib inline

# Basic DATA ##########################################################################################################
#Image data
root_folder = os.path.abspath("")
train_dir = os.path.join(root_folder, '0_traindata/')
baseimage = 'TIF'
codimage = '{}.'+ baseimage

#Train Data info
# Basic DAta: Train points/polygons data
trainxy_folder = os.path.abspath("1_trainshapes/")

modelo = 'modelo'
gridimages = modelo + '_' + 'grid_extent_full.csv'
predimages = modelo + '_' + 'predict_extent_full.csv'
fishnetfile = modelo + '_' + 'fishnet_grid_extent.csv'
trainxy = modelo + '_' + 'xy_deadtree.csv'


#Template Matching Output
tempmatch_out_folder = os.path.abspath("y_outCNN/")


print(tempmatch_out_folder)

C:\Users\snojek\Documents\0-Model_Development\1_MdP_Plantacion\y_outCNN


In [2]:
# Lets find the circles/trees

#Carga imagen Base
def loadimagefase2(IMagenID):
    print("Process 1 - Load image: ", IMagenID)
    #Load bands
    name = 'y_outCNN/fase1_' + str(IMagenID) + '_v0.tiff'
    
    cube = gdal.Open(name)
    bnd1 = cube.GetRasterBand(1)
    img1 = bnd1.ReadAsArray(0,0,cube.RasterXSize, cube.RasterYSize)
    
    #Stack an image to 3 bands/display
    img = np.array(img1, np.uint8)
    im_size = img.shape[:2]

   
    #Other way to get image corners...with GDAL
    ulx, xres, xskew, uly, yskew, yres  = cube.GetGeoTransform()
    lrx = ulx + (cube.RasterXSize * xres)
    lry = uly + (cube.RasterYSize * yres)

    xfull_max = float(lrx)
    xfull_min = float(ulx)
    yfull_max = float(uly)
    yfull_min = float(lry)

    print("XMax", xfull_max, "XMin", xfull_min, "YMax", yfull_max, "YMin", yfull_min)
    print("Input image size: ", im_size)
    print("Process 1 - DONE")
    
    return img, im_size, xfull_max, xfull_min, yfull_max, yfull_min
    

def get_scalers(im_size, xfull_max, xfull_min, yfull_max, yfull_min):
    h, w = im_size
    w_ = 1 * w * w / (w + 1)
    h_ = 1 * h * h / (h + 1)
    wi_max = abs(xfull_max - xfull_min)
    he_max = abs(yfull_max - yfull_min)
    return w_ / wi_max, h_ / he_max, w, h
    


###################################################################################
# Set up the SimpleBlobdetector with default parameters.
def simpleblob():
    params = cv2.SimpleBlobDetector_Params()

    # Change thresholds
    params.minThreshold = 0
    params.maxThreshold = 250

    # Filter by Area.
    params.filterByArea = True
    params.minArea = 12
    #params.maxArea = 10000

    # Filter by Circularity
    params.filterByCircularity = True
    params.minCircularity = 0.05

    # Filter by Convexity
    params.filterByConvexity = True
    params.minConvexity = 0.05

    # Filter by Inertia
    params.filterByInertia =True
    params.minInertiaRatio = 0.05

    detector = cv2.SimpleBlobDetector_create(params)
    return detector


def detectblobs(img, detector, xfull_min, yfull_max, x_scaler, y_scaler):
    #draw shapefile limits
    print(img.shape)
    smooth_rodal = 0
    rod_xfull_max = xfull_max - 5
    rod_xfull_min = xfull_min + 5
    
    rod_yfull_max = yfull_max - 5
    rod_yfull_min = yfull_min + 5
    
    a = int((yfull_max-rod_yfull_max)*y_scaler) - smooth_rodal
    b = a + int((rod_yfull_max-rod_yfull_min)*y_scaler) + smooth_rodal
    c = int((rod_xfull_min-xfull_min)*x_scaler) - smooth_rodal
    d = c + int((rod_xfull_max-rod_xfull_min)*x_scaler) + smooth_rodal

    print(a,b,c ,d)

    count = 1
    
    #Set grid limits
    lim = img.shape[0]
    lim1 = int(lim/10000) + 1
    lim1 = int(lim/6000) + 1
    lim2 = lim1
    
    smooth_micro = 10
    xkey = []
    for i in range (lim1):
        for j in range (lim2):
            a1 = int(a + i * ((b-a)/lim1))
            a2 = int(a + (i+1)*((b-a)/lim1) + smooth_micro)
            b1 = int(c + j * ((d-c)/lim2))
            b2 = int(c + (j+1)*((d-c)/lim2) + smooth_micro)
            print("Lote", count, "de",lim1*lim2 ,"limites", a1, ":",a2,",",b1,":",b2)

            imatarget = img[a1:a2,b1:b2]

            # Detect blobs.
            reversemask=255-imatarget
            #reversemask=ima3

            ###BLOB###
            keypoints = detector.detect(reversemask)
            xkey.append(keypoints)
            print("Keypoints: ",len(keypoints))
            count = count + 1
    
    return xkey, rod_xfull_max, rod_xfull_min, rod_yfull_max, rod_yfull_min, lim1, lim2

              
###########################################################################################################
#We are goint to write a file with the centroids.
#First convert XKEY to one array
######################################################################################################
def mask_for_polygons(points, radio, y_scaler, x_scaler, rod_yfull_min, rod_yfull_max, rod_xfull_min, rod_xfull_max, lim1, lim2):
    img_mask = np.zeros(im_size, np.uint8)

    #draw shapefile limits
    smooth = 0
    a = int((rod_yfull_min-yfull_min)*y_scaler) - smooth
    b = a + int((rod_yfull_max-rod_yfull_min)*y_scaler) + smooth
    c = int((rod_xfull_min-xfull_min)*x_scaler) - smooth
    d = c + int((rod_xfull_max-rod_xfull_min)*x_scaler) + smooth

    sizey = int((b - a)/lim1)
    sizex = int((d - c)/lim2)
    print("sizey: ", sizey, " sizeX: ", sizex)
    
    count = 0
    for x in range (lim2):
        for y in range (lim1):
            j = count
            count = count +1
           
            a2 = x * sizey
            b2 = y * sizex
            
            print(a, b)
            
            for i in range (len(points[j])):
                ycoord = (int(xkey[j][i].pt[0]))+b2
                xcoord = (int(xkey[j][i].pt[1]))+a2
                
                cv2.circle(img_mask, (ycoord, xcoord), radio, 255)
                cv2.circle(img_mask, (ycoord, xcoord), radio-1, 255)
                cv2.circle(img_mask, (ycoord, xcoord), radio-2, 255)
                #cv2.circle(img_mask, (ycoord, xcoord), radio-3, 255)        
                #cv2.circle(img_mask, (ycoord, xcoord), radio-4, 255) 
                #cv2.circle(img_mask, (ycoord, xcoord), radio-5, 255)
            print(len(points))
    return img_mask, a, b, c, d



def savekeytocoord(IMagenID, keytocoord, xfull_min,yfull_min,xfull_max,yfull_max, rod_yfull_max, scr):
    array = keytocoord

    xmin,ymin,xmax,ymax = [xfull_min,yfull_min,xfull_max,yfull_max]
    print(array.shape)
    
    nrows = array.shape[0]
    ncols = array.shape[1]
    xres = (xmax-xmin)/float(ncols)
    yres = (ymax-ymin)/float(nrows)

    dify = rod_yfull_max - yfull_max
    geotransform=(xmin,xres,0,ymax,0, -yres)
    # That's (top left x, w-e pixel resolution, rotation (0 if North is up), 
    #         top left y, rotation (0 if North is up), n-s pixel resolution)
    # I don't know why rotation is in twice???

    name = 'z_outFase3/'+ 'fase2_' + IMagenID + '.tif'
    output_raster = gdal.GetDriverByName('GTiff').Create(name, ncols, nrows, 1 ,gdal.GDT_Byte)  # Open the file
    output_raster.SetGeoTransform(geotransform)  # Specify its coordinates
    srs = osr.SpatialReference()                 # Establish its coordinate encoding
    srs.ImportFromEPSG(scr)                     # This one specifies WGS84 lat long.
                                                 # Anyone know how to specify the 
                                                 # IAU2000:49900 Mars encoding?
    output_raster.SetProjection( srs.ExportToWkt() )   # Exports the coordinate system 
                                                       # to the file
    output_raster.GetRasterBand(1).WriteArray(array)   # Writes my array to the raster

    output_raster = None


#############################################################################################################################
# Save centroids to array CSV file
def save_centroids(points, a, b, c, d, lim1, lim2, x_scaler, y_scaler):
    #save to xy file
    centroids = []
    centroids2 = []

    sizey = int((b - a)/lim1)
    sizex = int((d - c)/lim2)
   
    count = 0
    for x in range (lim2):
        for y in range (lim1):
            j = count
            count = count +1
           
            a2 = x * sizey
            b2 = y * sizex
            
            print(a, b)
            
            for i in range (len(points[j])):
                ycoord = (int(xkey[j][i].pt[0]))+b2
                xcoord = (int(xkey[j][i].pt[1]))+a2

                centroids.append((xcoord, ycoord))                    
    
    smooth = 0
    a = (((yfull_max-rod_yfull_max)*1) - smooth/y_scaler)
    c = (((rod_xfull_min-xfull_min)*1) - smooth/x_scaler)
    
    for i in range (len(centroids)):

        ycoord1 = -(centroids[i][0] - float(h)) 

        ycoord2 = float(ycoord1 / (y_scaler))  
        xcoord1 = float(centroids[i][1] /  (x_scaler)) 

        ycoord3 = abs(ycoord2 + float(yfull_min) - a)
        xcoord2 = abs(xcoord1 + float(xfull_min) + c)            
        
        centroids2.append((xcoord2, ycoord3)) 
            
    return centroids2

In [3]:
#Lets try to read the grid file and get all the images names

#Load the target images 
GS = pd.read_csv(trainxy_folder + '/' + predimages, names=['id', 'ImageId'], skiprows=1)
FS = pd.DataFrame(GS)
count = 1
first = 0

#Loop target images
for imagetarget in FS.iterrows():
    idimage = imagetarget[1][0]
    IMagenID = imagetarget[1][1].strip()
    print("Load data from: ", IMagenID, "Id: ", idimage)
    print("Image: ", count, " from ", len(FS))
    
    img, im_size, xfull_max, xfull_min, yfull_max, yfull_min = loadimagefase2(IMagenID)
    x_scaler, y_scaler, w, h = get_scalers(im_size, xfull_max, xfull_min, yfull_max, yfull_min)
    
    detector = simpleblob()
    xkey, rod_xfull_max, rod_xfull_min, rod_yfull_max, rod_yfull_min, lim1, lim2 = detectblobs(img, detector, xfull_min, yfull_max, x_scaler, y_scaler)

    keytocoord, a, b, c, d = mask_for_polygons(xkey, 3, y_scaler, x_scaler, rod_yfull_min, rod_yfull_max, rod_xfull_min, rod_xfull_max, lim1, lim2)
    nrows = keytocoord.shape[0]
    ncols = keytocoord.shape[1]
    M = np.float32([[1,0,c],[0,1,a]])
    #keytocoord = cv2.warpAffine(keytocoord,M,(ncols,nrows))

    #scr = 32720
    #savekeytocoord(IMagenID, keytocoord, xfull_min,yfull_min,xfull_max,yfull_max, rod_yfull_max, scr)

    centroidsarray = save_centroids(xkey, a, b, c, d, lim1, lim2, x_scaler, y_scaler)
    centroids_name = 'z_outFase3/' + 'centroids_' + IMagenID +'_v0.csv'
    np.savetxt(centroids_name, centroidsarray, fmt='%f', header="X, Y", delimiter=",")

    count = count + 1            
    print("-------------------------------------------------------------------------------------")

print(" ## Finish prediction ##")

Load data from:  El_Lucero_II_dam_JIB_utm_10cm Id:  121
Image:  1  from  1
Process 1 - Load image:  El_Lucero_II_dam_JIB_utm_10cm
XMax 470954.93168000004 XMin 469847.33168000006 YMax 6306811.99179 YMin 6305106.99179
Input image size:  (17050, 11076)
Process 1 - DONE
(17050, 11076)
49 16998 49 11024
Lote 1 de 9 limites 49 : 5708 , 49 : 3717
Keypoints:  13187
Lote 2 de 9 limites 49 : 5708 , 3707 : 7375
Keypoints:  8606
Lote 3 de 9 limites 49 : 5708 , 7365 : 11034
Keypoints:  7355
Lote 4 de 9 limites 5698 : 11358 , 49 : 3717
Keypoints:  8524
Lote 5 de 9 limites 5698 : 11358 , 3707 : 7375
Keypoints:  10341
Lote 6 de 9 limites 5698 : 11358 , 7365 : 11034
Keypoints:  7694
Lote 7 de 9 limites 11348 : 17008 , 49 : 3717
Keypoints:  1351
Lote 8 de 9 limites 11348 : 17008 , 3707 : 7375
Keypoints:  8135
Lote 9 de 9 limites 11348 : 17008 , 7365 : 11034
Keypoints:  948
sizey:  5649  sizeX:  3658
49 16998
9
49 16998
9
49 16998
9
49 16998
9
49 16998
9
49 16998
9
49 16998
9
49 16998
9
49 16998
9
49 169